# Direction 3 Multi-Dataset Colab Runner

This version is restart-safe for free-tier Colab.

What changed:
- the repository lives inside Google Drive instead of `/content`
- benchmark stages are split so CTGAN, TVAE, and the TABDDPM-aware registry flow can be checkpointed independently
- benchmark evaluation, plots, compute summaries, and reproducibility exports all persist in Drive automatically
- each major stage writes a zip export into Drive after it finishes
- you can reconnect later and continue from the same Drive-backed workspace

Recommended runtime: `T4 GPU`.
Do not run the benchmark or training cells on CPU unless you intentionally want a very slow run.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

WORKSPACE_ROOT = Path('/content/drive/MyDrive/direction3_multidataset_workspace')
REPO_DIR = WORKSPACE_ROOT / 'synthetic-data-lifecycle-benchmarks'
EXPORT_DIR = WORKSPACE_ROOT / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Workspace root:', WORKSPACE_ROOT)
print('Repo dir:', REPO_DIR)
print('Export dir:', EXPORT_DIR)


In [ ]:
import os
import subprocess
from pathlib import Path

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/Shubhamisl/synthetic-data-lifecycle-benchmarks.git', str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main'], check=True)

os.chdir(REPO_DIR)
print('Current working directory:', Path.cwd())


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab_direction3.txt'], check=True)


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU runtime not available. Reconnect to a T4 GPU before running benchmark or training cells.')


## Dataset Preparation

The benchmark datasets are prepared once and then reused across the staged runs.


In [ ]:
subprocess.run([sys.executable, '-m', 'data.loader'], check=True)
subprocess.run([sys.executable, '-m', 'benchmarks.download_datasets'], check=True)


## Optional Dry-Runs

Use these only to validate wiring. They do not train models.


In [ ]:
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'adult', '--dry-run'], check=True)
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'bank', '--dry-run'], check=True)
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'diabetes', '--dry-run'], check=True)
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'covertype', '--dry-run'], check=True)


## Archiving Helpers

These helpers write zip exports into Drive after each major stage. Even if Colab resets later, the exported zips remain in Drive.


In [ ]:
from contextlib import contextmanager
from datetime import datetime
import zipfile

import pandas as pd
from IPython.display import Image, display

from benchmarks import benchmark_models, evaluate_benchmarks, export_compute_summary, export_reproducibility, run_benchmarks, train_benchmark_models, visualize_benchmarks

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def archive_stage(stage_name: str, roots: list[Path]) -> Path:
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archive_path = EXPORT_DIR / f'{stage_name}_{stamp}.zip'
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root in roots:
            if not root.exists():
                print('Skipping missing path:', root)
                continue
            if root.is_file():
                zf.write(root, root.relative_to(REPO_DIR))
                continue
            for file_path in root.rglob('*'):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(REPO_DIR))
    print('Saved archive:', archive_path)
    return archive_path


def latest_exports():
    return sorted(EXPORT_DIR.glob('*.zip'))


@contextmanager
def _patched_trainable_model_ids(model_ids: tuple[str, ...]):
    original = benchmark_models.get_trainable_benchmark_model_ids
    benchmark_models.get_trainable_benchmark_model_ids = lambda: tuple(model_ids)
    try:
        yield
    finally:
        benchmark_models.get_trainable_benchmark_model_ids = original


def run_model_stage(model_id: str) -> bool:
    spec = benchmark_models.get_benchmark_model_spec(model_id)
    if not spec.trainable:
        print(f'Skipping {spec.display_name}: backend unavailable in this runtime.')
        return False

    with _patched_trainable_model_ids((model_id,)):
        train_benchmark_models.main()
    return True


## Cross-Domain Benchmark: CTGAN


In [ ]:
ctgan_ran = run_model_stage('ctgan')
ctgan_archive = archive_stage(
    'benchmark_ctgan',
    [
        REPO_DIR / 'benchmarks' / 'synthetic',
        REPO_DIR / 'benchmarks' / 'results',
    ],
)
ctgan_archive


## Cross-Domain Benchmark: TVAE


In [ ]:
tvae_ran = run_model_stage('tvae')
tvae_archive = archive_stage(
    'benchmark_tvae',
    [
        REPO_DIR / 'benchmarks' / 'synthetic',
        REPO_DIR / 'benchmarks' / 'results',
    ],
)
tvae_archive


## Cross-Domain Benchmark: TABDDPM-Aware Registry Flow

This stage only runs if the TABDDPM backend is available in the current runtime. That keeps the notebook from falling back to a slow or unsupported CPU path.


In [ ]:
tabddpm_ran = run_model_stage('tabddpm')
tabddpm_archive = archive_stage(
    'benchmark_tabddpm_registry_flow',
    [
        REPO_DIR / 'benchmarks' / 'synthetic',
        REPO_DIR / 'benchmarks' / 'results',
    ],
)
tabddpm_archive


## Benchmark Evaluation and Plots

After the staged training cells finish, this pass refreshes the cross-domain summary tables and the paper-facing plots.


In [ ]:
evaluate_benchmarks.main()
visualize_benchmarks.main()

summary_path = REPO_DIR / 'benchmarks' / 'results' / 'cross_domain_summary.csv'
rank_path = REPO_DIR / 'benchmarks' / 'results' / 'mean_rank_table.csv'

print('Cross-domain summary:')
display(pd.read_csv(summary_path))
print('Mean rank table:')
display(pd.read_csv(rank_path))

for image_path in [
    REPO_DIR / 'benchmarks' / 'plots' / 'plot1_tstr_heatmap.png',
    REPO_DIR / 'benchmarks' / 'plots' / 'plot2_cross_domain_dashboard.png',
    REPO_DIR / 'benchmarks' / 'plots' / 'plot3_mean_rank.png',
    REPO_DIR / 'benchmarks' / 'plots' / 'plot4_privacy_utility_all_domains.png',
]:
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))

benchmark_eval_archive = archive_stage(
    'benchmark_evaluation_and_plots',
    [
        REPO_DIR / 'benchmarks' / 'results',
        REPO_DIR / 'benchmarks' / 'plots',
    ],
)
benchmark_eval_archive


## Compute and Reproducibility Exports

These exports keep the reviewer-closure artifacts explicit and portable.


In [ ]:
compute_outputs = export_compute_summary.write_compute_summary()
repro_outputs = export_reproducibility.write_reproducibility_exports()

print('Compute summary:')
display(pd.read_csv(compute_outputs['csv']))
print('Reproducibility manifest:')
display(pd.read_csv(repro_outputs['reproducibility_manifest']))

compute_archive = archive_stage(
    'benchmark_compute_exports',
    [REPO_DIR / 'benchmarks' / 'results' / 'compute'],
)
repro_archive = archive_stage(
    'benchmark_reproducibility_exports',
    [REPO_DIR / 'benchmarks' / 'results' / 'reproducibility'],
)
compute_archive, repro_archive


## Full Direction 3 Runs

Run one dataset at a time. Each cell keeps the outputs in Drive and also creates a dataset-specific zip in Drive when the run finishes.


In [ ]:
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'adult', '--refresh-results'], check=True)
adult_archive = archive_stage('dp_triangle_adult', [REPO_DIR / 'results', REPO_DIR / 'models' / 'saved'])
adult_archive


In [ ]:
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'bank', '--refresh-results'], check=True)
bank_archive = archive_stage('dp_triangle_bank', [REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'bank'])
bank_archive


In [ ]:
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'diabetes', '--refresh-results'], check=True)
diabetes_archive = archive_stage('dp_triangle_diabetes', [REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'diabetes'])
diabetes_archive


In [ ]:
subprocess.run([sys.executable, '-m', 'dp_triangle.run_direction3', '--dataset', 'covertype', '--refresh-results'], check=True)
covertype_archive = archive_stage('dp_triangle_covertype', [REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'covertype'])
covertype_archive


## Final Combined Archive

This creates one all-in-one zip in Drive after the per-stage runs are complete.


In [ ]:
combined_archive = EXPORT_DIR / f'direction3_multidataset_outputs_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
roots = [
    REPO_DIR / 'results',
    REPO_DIR / 'models' / 'saved',
    REPO_DIR / 'benchmarks' / 'results',
    REPO_DIR / 'benchmarks' / 'plots',
    REPO_DIR / 'benchmarks' / 'synthetic',
]

with zipfile.ZipFile(combined_archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root in roots:
        if not root.exists():
            print('Skipping missing path:', root)
            continue
        if root.is_file():
            zf.write(root, root.relative_to(REPO_DIR))
            continue
        for file_path in root.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(REPO_DIR))

print('Combined archive:', combined_archive)
combined_archive


In [ ]:
latest_exports()


## Resume After A Runtime Reset

If Colab disconnects later:
- reconnect the runtime
- remount Drive
- rerun the clone/update cell
- rerun the install cell if needed
- continue from the stage cell you had not finished yet

Because the repo and outputs live in Drive, previously completed stages should still be there.
